## LIBRARIES AND DECLARATION

In [1]:
import os
import numpy as np 
import pandas as pd 
from IPython.display import FileLink

## CONFIG

In [2]:
ID_COL     = 'id'
INPUT_DIR  = '/kaggle/input/datasets/dhuyent/classified-filtered-stage2'

GROUPS = ['logic1', 'logic2', 'reference1']
RUNS   = ['claude-opus-4-8', 'gpt-5.6-sol'] # model       
FNAME  = 'step1_classify2_{group}_{run}.csv'

# Columns to drop from the output
EXCLUDE_COLUMNS = [
    'llm_model', 'prompt_strategy', 'pred_status', 'pred_input',
    'pred_actual_output', 'pred_expected_output', 'pred_reason',
    'match', 'label'
]

LABEL_COL   = 'filtered_label' # label column after manual recheck
KEEP_LABELS = [2, 3]

## INTERSECTION FILTERING (2 MODELS)

Filter the `id`s that neither model judged correctly.

For each group (`logic1`, `logic2`, `reference1`), take the intersection of `id`s labeled 2 or 3 across both models (`gpt-5.6-sol` and `claude-opus-4-8`):

- **Label 2**: SOME but not all attempts match (partial match)
- **Label 3**: NO attempt matches

An `id` is kept only if it falls in Label 2 or 3 for **both** models. `id`s that either model judged fully correct (Label 1) are dropped.

In [3]:
for group in GROUPS:
    for run in RUNS:
        path = os.path.join(INPUT_DIR, FNAME.format(group=group, run=run))
        df   = pd.read_csv(path)

        ids = df.loc[df[LABEL_COL].isin([2, 3]), ID_COL].sort_values().tolist()

        print(f'=== {group} | {run} ===')
        print(f'id (label 2/3): {len(ids)}')
        print(ids)
        print()

=== logic1 | claude-opus-4-8 ===
id (label 2/3): 25
[18, 32, 38, 40, 41, 45, 49, 60, 63, 72, 80, 91, 96, 108, 109, 110, 111, 127, 130, 132, 133, 134, 139, 152, 154]

=== logic1 | gpt-5.6-sol ===
id (label 2/3): 20
[38, 41, 45, 49, 60, 63, 72, 91, 96, 108, 109, 111, 113, 125, 130, 132, 133, 134, 172, 178]

=== logic2 | claude-opus-4-8 ===
id (label 2/3): 34
[6, 9, 19, 22, 23, 26, 35, 40, 41, 44, 45, 49, 51, 53, 56, 63, 66, 76, 78, 82, 83, 85, 103, 111, 119, 123, 125, 128, 133, 134, 142, 153, 158, 172]

=== logic2 | gpt-5.6-sol ===
id (label 2/3): 24
[6, 23, 26, 35, 41, 44, 49, 51, 53, 76, 78, 82, 83, 85, 89, 103, 111, 119, 123, 133, 153, 158, 160, 172]

=== reference1 | claude-opus-4-8 ===
id (label 2/3): 44
[18, 20, 21, 23, 29, 35, 46, 53, 54, 57, 60, 61, 63, 66, 71, 72, 73, 76, 78, 80, 81, 96, 103, 105, 107, 110, 111, 112, 115, 116, 117, 118, 124, 125, 126, 127, 133, 135, 139, 140, 141, 148, 164, 172]

=== reference1 | gpt-5.6-sol ===
id (label 2/3): 35
[18, 20, 29, 35, 46, 53, 57, 60

In [4]:
def combine_group(group):
    files = [os.path.join(INPUT_DIR, FNAME.format(group=group, run=r)) for r in RUNS]
    dfs   = [pd.read_csv(f) for f in files]

    # Lọc label trước khi giao
    if KEEP_LABELS is not None:
        dfs = [d[d[LABEL_COL].isin(KEEP_LABELS)] for d in dfs]

    id_sets = [set(d[ID_COL]) for d in dfs]
    common  = set.intersection(*id_sets)  # phép giao

    df_common = (dfs[0][dfs[0][ID_COL].isin(common)]
                 .sort_values(ID_COL).reset_index(drop=True))

    if EXCLUDE_COLUMNS:
        dropped   = [c for c in EXCLUDE_COLUMNS if c in df_common.columns]
        df_common = df_common.drop(columns=dropped)

    out = f'step1_filter2_{group}.csv'
    df_common.to_csv(out, index=False, encoding='utf-8-sig')

    detail = ' | '.join(f'{r}={len(s)}' for r, s in zip(RUNS, id_sets))
    print(f'[{group}] {detail} -> {len(common)} ids in intersection | Saved to {out}')
    return out

## SAVE OUTPUT

In [5]:
for g in GROUPS:
    path = combine_group(g)
    display(FileLink(path))

[logic1] claude-opus-4-8=25 | gpt-5.6-sol=20 -> 16 ids in intersection | Saved to step1_filter2_logic1.csv


/kaggle/working/step1_filter2_logic1.csv

[logic2] claude-opus-4-8=34 | gpt-5.6-sol=24 -> 22 ids in intersection | Saved to step1_filter2_logic2.csv


/kaggle/working/step1_filter2_logic2.csv

[reference1] claude-opus-4-8=44 | gpt-5.6-sol=35 -> 35 ids in intersection | Saved to step1_filter2_reference1.csv


/kaggle/working/step1_filter2_reference1.csv